Bước 1: Khởi tạo SparkSession và đọc bộ dữ liệu thương mại điện tử từ HDFS.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Ecommerce_Category_Revenue_LinearRegression") \
    .getOrCreate()

HDFS_PATH = "hdfs://localhost:9000/ecom/Pakistan_Largest_Ecommerce_Dataset.csv"

df = spark.read.csv(HDFS_PATH, header=True, inferSchema=False)

df.show(5, truncate=False)
df.printSchema()

+-------+--------------+----------+-----------------------------------------------------------+-----+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
|item_id|status        |created_at|sku                                                        |price|qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|Working Date|BI Status| MV    |Year|Month|Customer Since|M-Y   |FY  |Customer ID|_c21|_c22|_c23|_c24|_c25|
+-------+--------------+----------+-----------------------------------------------------------+-----+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+----+----+----+----+----+
|211131 |complete      |7/1/2016  |kreations_YI 06-L

Bước 2: Xem số dòng và số cột ban đầu

In [2]:
num_rows = df.count()
num_cols = len(df.columns)

print("Số dòng ban đầu:", num_rows)
print("Số cột ban đầu:", num_cols)

print("Danh sách cột:")
print(df.columns)

Số dòng ban đầu: 1048586
Số cột ban đầu: 26
Danh sách cột:
['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'Working Date', 'BI Status', ' MV ', 'Year', 'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID', '_c21', '_c22', '_c23', '_c24', '_c25']


Bước 3: Xóa cột + dòng trắng

In [3]:
# Xóa cột trắng kiểu _c21, _c22...
df1 = df.drop(*[c for c in df.columns if c.startswith("_c")])

# Xóa dòng trắng
df1 = df1.dropna(how="all")

print("Số dòng sau xóa dòng trắng:", df1.count())
print("Số cột sau xóa cột trắng:", len(df1.columns))
print(df1.columns)

df1.show(5, truncate=False)

Số dòng sau xóa dòng trắng: 584535
Số cột sau xóa cột trắng: 21
['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'Working Date', 'BI Status', ' MV ', 'Year', 'Month', 'Customer Since', 'M-Y', 'FY', 'Customer ID']
+-------+--------------+----------+-----------------------------------------------------------+-----+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+------------+---------+-------+----+-----+--------------+------+----+-----------+
|item_id|status        |created_at|sku                                                        |price|qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|Working Date|BI Status| MV    |Year|Month|Customer Since|M-Y   |FY  |Customer ID|
+-------+--------------+----------+----------------------------------------

Bước 4: Xóa các cột không cần thiết và chuẩn hóa tên cột

In [4]:
cols_to_drop = [
    "Working Date",
    "BI Status",
    "MV",
    " MV ",
    "Year",
    "Month",
    "Customer Since",
    "M-Y",
    "FY"
]

existing_drop_cols = [c for c in cols_to_drop if c in df.columns]

df = df1.drop(*existing_drop_cols)

df = df.withColumnRenamed("Customer ID", "customer_id")

print("Số dòng:", df.count())
print("Số cột:", len(df.columns))
print(df.columns)

df.show(5, truncate=False)

Số dòng: 584535
Số cột: 13
['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'customer_id']
+-------+--------------+----------+-----------------------------------------------------------+-----+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+-----------+
|item_id|status        |created_at|sku                                                        |price|qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|customer_id|
+-------+--------------+----------+-----------------------------------------------------------+-----+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+-----------+
|211131 |complete      |7/1/2016  |kreations_YI 06-L                                          |1950 |1          |

Bước 5: Chuyển các giá trị "\N" thành None

In [5]:
import pyspark.sql.functions as F

print("Số dòng trước khi xử lý \\N:", df.count())

for c in df.columns:
    df = df.withColumn(
        c,
        F.when(F.col(c) == "\\N", None).otherwise(F.col(c))
    )

print("Số dòng sau khi xử lý \\N:", df.count())

Số dòng trước khi xử lý \N: 584535
Số dòng sau khi xử lý \N: 584535


Bước 6: Xóa đi các dòng có giá trị None trong các cột quan trọng là category_name_1, customer_id, grand_total, created_at, status

In [6]:
important_cols = [
    "category_name_1",
    "customer_id",
    "grand_total",
    "created_at",
    "status"
]

print("Số dòng trước khi xóa dòng thiếu cột quan trọng:", df.count())

df = df.dropna(subset=important_cols)

print("Số dòng sau khi xóa dòng thiếu cột quan trọng:", df.count())

Số dòng trước khi xóa dòng thiếu cột quan trọng: 584535
Số dòng sau khi xóa dòng thiếu cột quan trọng: 576484


Bước 7: Với các cột không quan trọng, các giá trị None sẽ được chuyển thành "unknown"

In [7]:
print("Số dòng trước khi fill null cột phụ:", df.count())

df = df.fillna({
    "sales_commission_code": "unknown",
    "payment_method": "unknown"
})

print("Số dòng sau khi fill null cột phụ:", df.count())

Số dòng trước khi fill null cột phụ: 576484
Số dòng sau khi fill null cột phụ: 576484


Bước 8: Xóa bỏ các dòng bị trùng lặp

In [8]:
print("Số dòng trước khi xóa trùng hoàn toàn:", df.count())

df = df.dropDuplicates()

print("Số dòng sau khi xóa trùng hoàn toàn:", df.count())

Số dòng trước khi xóa trùng hoàn toàn: 576484
Số dòng sau khi xóa trùng hoàn toàn: 576484


Bước 9: Chuẩn hóa dữ liệu số về dạng double để tính toán về sau

In [9]:
import pyspark.sql.functions as F

def clean_double(c):
    return F.regexp_replace(F.col(c).cast("string"), ",", "").cast("double")

print("Số dòng trước khi làm sạch cột số:", df.count())

df = df.withColumn("price", clean_double("price"))
df = df.withColumn("qty_ordered", clean_double("qty_ordered"))
df = df.withColumn("grand_total", clean_double("grand_total"))
df = df.withColumn("discount_amount", clean_double("discount_amount"))

print("Số dòng sau khi làm sạch cột số:", df.count())

df.select("price", "qty_ordered", "grand_total", "discount_amount").show(10)

Số dòng trước khi làm sạch cột số: 576484
Số dòng sau khi làm sạch cột số: 576484
+------+-----------+-----------+---------------+
| price|qty_ordered|grand_total|discount_amount|
+------+-----------+-----------+---------------+
|4950.0|        1.0|     4950.0|            0.0|
| 360.0|        1.0|      360.0|            0.0|
| 240.0|        1.0|      240.0|            0.0|
| 320.0|        1.0|      320.0|            0.0|
|3150.0|        1.0|     3150.0|            0.0|
| 490.0|        1.0|      490.0|            0.0|
| 144.0|        1.0|     1879.0|            0.0|
| 680.0|        1.0|      680.0|            0.0|
| 999.0|        1.0|      999.0|            0.0|
| 140.0|        1.0|      140.0|            0.0|
+------+-----------+-----------+---------------+
only showing top 10 rows


Bước 10: Xóa đi các giá trị vô lý

In [10]:
print("Số dòng trước khi xóa giá trị số vô lý:", df.count())

df = df.filter(
    (F.col("price") >= 0) &
    (F.col("qty_ordered") > 0) &
    (F.col("grand_total") >= 0) &
    (F.col("discount_amount") >= 0)
)

print("Số dòng sau khi xóa giá trị số vô lý:", df.count())

Số dòng trước khi xóa giá trị số vô lý: 576484
Số dòng sau khi xóa giá trị số vô lý: 576406


Bước 11: Xử lý ngày tháng và tạo thêm cột tháng, năm để có thể phân tích xu hướng ở các bước sau

In [11]:
import pyspark.sql.functions as F

print("Số dòng trước khi xử lý created_at:", df.count())

df = df.withColumn(
    "created_date",
    F.coalesce(
        F.expr("try_cast(to_timestamp(created_at, 'M/d/yyyy') as date)"),
        F.expr("try_cast(to_timestamp(created_at, 'MM/dd/yyyy') as date)"),
        F.expr("try_cast(to_timestamp(created_at, 'M/d/yy') as date)"),
        F.expr("try_cast(to_timestamp(created_at, 'MM/dd/yy') as date)")
    )
)

df = df.withColumn("order_year", F.year("created_date"))
df = df.withColumn("order_month", F.month("created_date"))

df = df.filter(F.col("created_date").isNotNull())

print("Số dòng sau khi xử lý created_at:", df.count())

df.select("created_at", "created_date", "order_year", "order_month").show(10, truncate=False)

Số dòng trước khi xử lý created_at: 576406
Số dòng sau khi xử lý created_at: 576406
+----------+------------+----------+-----------+
|created_at|created_date|order_year|order_month|
+----------+------------+----------+-----------+
|7/1/2016  |2016-07-01  |2016      |7          |
|7/1/2016  |2016-07-01  |2016      |7          |
|7/1/2016  |2016-07-01  |2016      |7          |
|7/1/2016  |2016-07-01  |2016      |7          |
|7/2/2016  |2016-07-02  |2016      |7          |
|7/4/2016  |2016-07-04  |2016      |7          |
|7/9/2016  |2016-07-09  |2016      |7          |
|7/9/2016  |2016-07-09  |2016      |7          |
|7/11/2016 |2016-07-11  |2016      |7          |
|7/11/2016 |2016-07-11  |2016      |7          |
+----------+------------+----------+-----------+
only showing top 10 rows


Bước 12: Xem thống kê mô tả

In [12]:
import pyspark.sql.functions as F

numeric_cols = [
    "price",
    "qty_ordered",
    "grand_total",
    "discount_amount"
]

print("Số dòng hiện tại:", df.count())
print("Số cột hiện tại:", len(df.columns))

df.select(numeric_cols).describe().show(truncate=False)

Số dòng hiện tại: 576406
Số cột hiện tại: 16
+-------+------------------+------------------+------------------+------------------+
|summary|price             |qty_ordered       |grand_total       |discount_amount   |
+-------+------------------+------------------+------------------+------------------+
|count  |576406            |576406            |576406            |576406            |
|mean   |6381.108578831592 |1.2952224647210473|8587.954477332687 |503.2807171922307 |
|stddev |15007.600382685905|3.9979958250638696|61735.730758717735|1510.7408669557753|
|min    |0.0               |1.0               |0.0               |0.0               |
|max    |1012625.9         |1000.0            |1.7888E7          |90300.0           |
+-------+------------------+------------------+------------------+------------------+



Bước 13: Xử lý ngoại lai

Sử dụng hàm approxQuantile để tính các phân vị Q1, Q2, Q3, P95 và P99 cho các biến số. Mục đích là quan sát phân phối dữ liệu và phát hiện các giá trị bất thường. Trong đó, Q2 là trung vị, Q1 và Q3 cho biết khoảng phân bố chính của dữ liệu, còn P95 và P99 giúp nhận diện nhóm giá trị rất cao có khả năng là ngoại lai

In [13]:
for c in numeric_cols:
    q = df.approxQuantile(c, [0.25, 0.5, 0.75, 0.95, 0.99], 0.01)

    print("Cột:", c)
    print("Q1  :", q[0])
    print("Q2  :", q[1])
    print("Q3  :", q[2])
    print("P95 :", q[3])
    print("P99 :", q[4])
    print("-" * 40)

Cột: price
Q1  : 360.0
Q2  : 899.0
Q3  : 3938.0
P95 : 28250.0
P99 : 1012625.9
----------------------------------------
Cột: qty_ordered
Q1  : 1.0
Q2  : 1.0
Q3  : 1.0
P95 : 2.0
P99 : 1000.0
----------------------------------------
Cột: grand_total
Q1  : 948.0
Q2  : 1949.0
Q3  : 6628.3
P95 : 29858.4
P99 : 17888000.0
----------------------------------------
Cột: discount_amount
Q1  : 0.0
Q2  : 0.0
Q3  : 149.7523
P95 : 2609.0
P99 : 90300.0
----------------------------------------


Nhóm sử dụng phương pháp IQR để xác định giá trị ngoại lai ở các biến số. Với mỗi biến, ngưỡng trên được tính bằng Q3 + 1.5*IQR; các dòng có giá trị vượt ngưỡng này được xem là ngoại lai và được thống kê để phục vụ bước xử lý dữ liệu tiếp theo.

In [14]:
outlier_cols = [
    "price",
    "qty_ordered",
    "grand_total"
]

for c in outlier_cols:
    q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr

    outlier_count = df.filter(F.col(c) > upper).count()

    print("Cột:", c)
    print("Upper bound:", upper)
    print("Số dòng ngoại lai:", outlier_count)
    print("-" * 40)

Cột: price
Upper bound: 9305.0
Số dòng ngoại lai: 107046
----------------------------------------
Cột: qty_ordered
Upper bound: 1.0
Số dòng ngoại lai: 78434
----------------------------------------
Cột: grand_total
Upper bound: 15148.75
Số dòng ngoại lai: 81710
----------------------------------------


Xóa đi các dòng có giá trị ngoại lai

In [15]:
def remove_outlier_iqr_report(df_input, col_name):
    before = df_input.count()

    q1, q3 = df_input.approxQuantile(col_name, [0.25, 0.75], 0.01)
    iqr = q3 - q1

    lower = 0
    upper = q3 + 1.5 * iqr

    df_output = df_input.filter(
        (F.col(col_name) >= lower) &
        (F.col(col_name) <= upper)
    )

    after = df_output.count()

    print("Cột:", col_name)
    print("Lower:", lower)
    print("Upper:", upper)
    print("Số dòng trước:", before)
    print("Số dòng sau:", after)
    print("Số dòng bị xóa:", before - after)
    print("-" * 40)

    return df_output

df = remove_outlier_iqr_report(df, "qty_ordered")
df = remove_outlier_iqr_report(df, "grand_total")

Cột: qty_ordered
Lower: 0
Upper: 1.0
Số dòng trước: 576406
Số dòng sau: 497972
Số dòng bị xóa: 78434
----------------------------------------
Cột: grand_total
Lower: 0
Upper: 15930.0
Số dòng trước: 497972
Số dòng sau: 429296
Số dòng bị xóa: 68676
----------------------------------------


XEM LẠI DATASET ĐÃ SẠCH

In [16]:
print("Số dòng:", df.count())
print("Số cột:", len(df.columns))

print("Danh sách cột:")
print(df.columns)

print("5 dòng đầu:")
df.show(5, truncate=False)

Số dòng: 429296
Số cột: 16
Danh sách cột:
['item_id', 'status', 'created_at', 'sku', 'price', 'qty_ordered', 'grand_total', 'increment_id', 'category_name_1', 'sales_commission_code', 'discount_amount', 'payment_method', 'customer_id', 'created_date', 'order_year', 'order_month']
5 dòng đầu:
+-------+--------------+----------+-----------------------------------------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+-----------+------------+----------+-----------+
|item_id|status        |created_at|sku                                                        |price |qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|customer_id|created_date|order_year|order_month|
+-------+--------------+----------+-----------------------------------------------------------+------+-----------+-----------+------------+-----------------+---------------------+----------

Kiểm tra giá trị null còn sót

In [17]:
important_cols = [
    "item_id",
    "status",
    "sku",
    "price",
    "qty_ordered",
    "grand_total",
    "increment_id",
    "category_name_1",
    "discount_amount",
    "payment_method",
    "customer_id",
    "created_date",
    "order_year",
    "order_month"
]

existing_cols = [c for c in important_cols if c in df.columns]

df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in existing_cols
]).show(truncate=False)

+-------+------+---+-----+-----------+-----------+------------+---------------+---------------+--------------+-----------+------------+----------+-----------+
|item_id|status|sku|price|qty_ordered|grand_total|increment_id|category_name_1|discount_amount|payment_method|customer_id|created_date|order_year|order_month|
+-------+------+---+-----+-----------+-----------+------------+---------------+---------------+--------------+-----------+------------+----------+-----------+
|0      |0     |0  |0    |0          |0          |0           |0              |0              |0             |0          |0           |0         |0          |
+-------+------+---+-----+-----------+-----------+------------+---------------+---------------+--------------+-----------+------------+----------+-----------+



Kiểm tra lại thống kê mô tả sau khi loại bỏ giá trị ngoại lai

In [18]:
numeric_cols = [
    "price",
    "qty_ordered",
    "grand_total",
    "discount_amount"
]

existing_numeric_cols = [c for c in numeric_cols if c in df.columns]

df.select(existing_numeric_cols).describe().show(truncate=False)

+-------+------------------+-----------+------------------+-----------------+
|summary|price             |qty_ordered|grand_total       |discount_amount  |
+-------+------------------+-----------+------------------+-----------------+
|count  |429296            |429296     |429296            |429296           |
|mean   |2449.2836012681228|1.0        |3026.956868923791 |303.4339934930677|
|stddev |4237.191610242377 |0.0        |3631.3428499937904|881.7138905796593|
|min    |0.0               |1.0        |0.0               |0.0              |
|max    |163000.0          |1.0        |15928.05          |30819.23         |
+-------+------------------+-----------+------------------+-----------------+



TẢI LẠI DỮ LIỆU SẠCH LÊN HDFS

In [21]:
OUTPUT_PATH = "hdfs://localhost:9000/ecom/ecom_clean_final"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(OUTPUT_PATH)

print("Đã lưu dataset sạch vào:", OUTPUT_PATH)

Đã lưu dataset sạch vào: hdfs://localhost:9000/ecom/ecom_clean_final


In [22]:
spark.read.csv(
    "hdfs://localhost:9000/ecom/ecom_clean_final",
    header=True,
    inferSchema=True
).show(5, truncate=False)

+-------+--------+----------+-----------------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+-----------+------------+----------+-----------+
|item_id|status  |created_at|sku                                |price |qty_ordered|grand_total|increment_id|category_name_1  |sales_commission_code|discount_amount|payment_method|customer_id|created_date|order_year|order_month|
+-------+--------+----------+-----------------------------------+------+-----------+-----------+------------+-----------------+---------------------+---------------+--------------+-----------+------------+----------+-----------+
|211313 |complete|7/1/2016  |RS_Kaju Barfi                      |425.0 |1.0        |1125.0     |100147569   |Soghaat          |unknown              |0.0            |cod           |57         |2016-07-01  |2016      |7          |
|211630 |complete|7/1/2016  |HR_Pani Puri 360g                  |350.0 |1.0        |